# Analysis of Missing Mass Histogram

This notebook analyzes the `h_mmiss_all_weighted` histogram from the combined analysis ROOT file, focusing on identifying and characterizing the peaks near 0.93, 1.2, 1.55, and 1.68 GeV.

## 1. Import Required Libraries

In [16]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings('ignore')

# Set up publication-quality matplotlib settings
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['lines.linewidth'] = 1.5
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['grid.linewidth'] = 0.8

## 2. Load ROOT File and Extract Histogram

In [18]:
# Define the ROOT file path
root_file_path = "../output/plots/x60_4b/production_wfpi0/combined_branches_LH2_wfpi0.root"

# Open the ROOT file with uproot
try:
    root_file = uproot.open(root_file_path)
    print(f"Successfully opened ROOT file: {root_file_path}")
except FileNotFoundError:
    print(f"Error: File not found at {root_file_path}")
    print("Please adjust the path if necessary.")
    root_file = None

# List available histograms
if root_file:
    print("\nAvailable objects in ROOT file:")
    for key in root_file.keys():
        print(f"  {key}")
    
    # Extract the histogram
    hist = root_file["h_mmiss_all_weighted"]
    print(f"\nExtracted histogram: h_mmiss_all_weighted")
    print(f"Type: {type(hist)}")


Successfully opened ROOT file: ../output/plots/x60_4b/production_wfpi0/combined_branches_LH2_wfpi0.root

Available objects in ROOT file:
  physics;1


KeyInFileError: not found: 'h_mmiss_all_weighted' (with any cycle number)

    Available keys: 'physics;1'

in file ../output/plots/x60_4b/production_wfpi0/combined_branches_LH2_wfpi0.root

## 3. Inspect Histogram Properties

In [11]:
# Extract histogram data
if root_file:
    # Get bin contents (values) and bin edges
    values = hist.values()
    edges = hist.axes[0].edges()
    bin_centers = (edges[:-1] + edges[1:]) / 2
    bin_widths = edges[1:] - edges[:-1]
    
    print("Histogram Properties:")
    print(f"  Number of bins: {len(values)}")
    print(f"  Bin edges range: {edges[0]:.4f} to {edges[-1]:.4f}")
    print(f"  Total counts: {np.sum(values):.0f}")
    print(f"  Max bin content: {np.max(values):.2f}")
    print(f"  Min bin content: {np.min(values):.2f}")
    print(f"  Mean value: {np.sum(values * bin_centers) / np.sum(values):.4f}")
    
    # Store for later use
    hist_data = {
        'values': values,
        'edges': edges,
        'centers': bin_centers,
        'widths': bin_widths
    }

## 4. Create Publication Quality Plot

In [12]:
if root_file:
    # Create figure with high quality settings
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Plot histogram with step function
    ax.step(edges[:-1], values, where='mid', linewidth=2.5, color='#1f77b4', label='$h_{m_{miss}}$ (weighted)')
    ax.fill_between(edges[:-1], values, step='mid', alpha=0.2, color='#1f77b4')
    
    # Styling
    ax.set_xlabel('Missing Mass $m_{miss}$ (GeV/$c^2$)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Weighted Counts', fontsize=14, fontweight='bold')
    ax.set_title('Missing Mass Distribution - All Events', fontsize=16, fontweight='bold', pad=20)
    
    # Grid
    ax.grid(True, which='major', alpha=0.3, linestyle='-', linewidth=0.8)
    ax.grid(True, which='minor', alpha=0.15, linestyle=':', linewidth=0.5)
    ax.minorticks_on()
    
    # Set reasonable axis limits
    ax.set_xlim(0, np.max(edges))
    ax.set_ylim(0, np.max(values) * 1.1)
    
    # Add legend
    ax.legend(loc='upper right', framealpha=0.95, edgecolor='black', fancybox=False)
    
    # Spine styling
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
    
    plt.tight_layout()
    # plt.savefig('mmiss_histogram_plot.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Publication-quality plot saved as 'mmiss_histogram_plot.png'")


## 5. Analyze Peak Positions

In [13]:
if root_file:
    # Find peaks using scipy
    # Adjust prominence and distance based on histogram characteristics
    peaks, properties = find_peaks(values, prominence=np.max(values)*0.05, distance=5)
    
    # Convert peak indices to mass values
    peak_masses = bin_centers[peaks]
    peak_values = values[peaks]
    
    # Sort peaks by mass value
    sort_idx = np.argsort(peak_masses)
    peak_masses_sorted = peak_masses[sort_idx]
    peak_values_sorted = peak_values[sort_idx]
    
    print("Detected Peaks in Missing Mass Histogram:")
    print("=" * 50)
    print(f"{'Peak #':<8} {'Mass (GeV)':<15} {'Bin Content':<15}")
    print("-" * 50)
    for i, (mass, value) in enumerate(zip(peak_masses_sorted, peak_values_sorted)):
        print(f"{i+1:<8} {mass:<15.4f} {value:<15.2f}")
    
    print("\n\nExpected Peaks (from observation):")
    expected_peaks = [0.93, 1.2, 1.55, 1.68]  # GeV
    print(f"{'Expected (GeV)':<20} Likely Resonance")
    print("-" * 50)
    print(f"{0.93:<20.2f} η (Eta meson)")
    print(f"{1.2:<20.2f} ω (Omega) / b₁")
    print(f"{1.55:<20.2f} f₁")
    print(f"{1.68:<20.2f} φ (Phi)")
    
    # Store peak data
    peak_data = {
        'masses': peak_masses_sorted,
        'values': peak_values_sorted,
        'indices': peaks[sort_idx]
    }


## 6. Annotate Peaks with Resonance Labels

In [15]:
if root_file:
    # Create annotated version with peak labels
    fig, ax = plt.subplots(figsize=(11, 6))
    
    # Plot histogram
    ax.step(edges[:-1], values, where='mid', linewidth=2.5, color='#1f77b4', label='Missing Mass')
    ax.fill_between(edges[:-1], values, step='mid', alpha=0.2, color='#1f77b4')
    
    # Plot detected peaks
    # ax.scatter(peak_masses_sorted, peak_values_sorted, color='red', s=150, marker='o', 
    #            zorder=5, edgecolors='darkred', linewidth=2, label='Detected Peaks')
    
    # Define resonance information based on PDG particle data
    resonances = [
        {'mass': 0.938, 'label': 'Proton', 'y_offset': 0.15},
        {'mass': 1.2, 'label': '$\\Delta$(1232)', 'y_offset': 0.25},
        {'mass': 1.52, 'label': '$N$(1520)/$N$(1535)', 'y_offset': 0.15},
        {'mass': 1.68, 'label': '$N$(1675)/$\\Delta$(1700)', 'y_offset': 0.25}
    ]
    
    # Add vertical lines and labels for expected resonances
    colors_expected = ['#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    for i, res in enumerate(resonances):
        # Vertical line
        ax.axvline(res['mass'], color=colors_expected[i], linestyle='--', 
                   linewidth=1.5, alpha=0.7, zorder=1)
        
        # Add mass value inside the plot (horizontal text)
        ax.text(res['mass'], np.max(values) * 0.02, f"{res['mass']:.3f}", 
                fontsize=10, fontweight='bold', ha='center', va='bottom', 
                rotation=0, color=colors_expected[i])
        
        # Annotation
        y_pos = np.max(values) * res['y_offset']
        ax.annotate(res['label'], xy=(res['mass'], y_pos), 
                   xytext=(res['mass'], y_pos),
                   fontsize=11, fontweight='bold', ha='center',
                   bbox=dict(boxstyle='round,pad=0.5', facecolor=colors_expected[i], 
                            alpha=0.3, edgecolor=colors_expected[i], linewidth=1.5),
                   arrowprops=dict(arrowstyle='->', color=colors_expected[i], lw=1.5))
    
    # Styling
    ax.set_xlabel('Missing Mass $m_{miss}$ (GeV/$c^2$)', fontsize=16, fontweight='bold')
    ax.set_ylabel('Counts', fontsize=16, fontweight='bold')
    ax.set_title('Missing Mass Distribution (KinC_x60_4b); LH2', fontsize=17, fontweight='bold', pad=20)
    
    # Grid
    ax.grid(True, which='major', alpha=0.3, linestyle='-', linewidth=0.8)
    ax.grid(True, which='minor', alpha=0.15, linestyle=':', linewidth=0.5)
    ax.minorticks_on()
    
    # Set axis limits
    ax.set_xlim(0, np.max(edges))
    ax.set_ylim(0, np.max(values) * 1.25)
    
    # Add legend
    ax.legend(loc='upper right', framealpha=0.95, edgecolor='black', fancybox=False, fontsize=12)
    
    # Spine styling
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
    
    plt.tight_layout()
    # plt.savefig('mmiss_histogram_annotated.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nAnnotated publication-quality plot saved as 'mmiss_histogram_annotated.png'")
    print("✓ Analysis complete! Both plots ready for publication use.")